In [18]:
from pathlib import Path

from functools import partial

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from PIL import Image

import timm

import re

from transformers import AutoModelForCausalLM
from transformers import AutoModel, AutoTokenizer
import torchmetrics
import torchvision.transforms as T
import albumentations as A

import pandas as pd
import numpy as np

In [19]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

In [20]:
BASE_DIR = Path.cwd().parent
print(BASE_DIR)

/home/evgeniy/Документы/GitHub/YandexNN/sprint_4


In [21]:
dataset_path = BASE_DIR / "data" / "multimodal"
images_path = dataset_path / "images"
df_path = dataset_path / "items.csv"

In [22]:
class Config:
    # Модели
    TEXT_MODEL_NAME = "bert-base-uncased"
    IMAGE_MODEL_NAME = "resnet50"

    # Какие слои размораживаем - совпадают с неймингом в моделях
    TEXT_MODEL_UNFREEZE = "encoder.layer.11|pooler"  
    IMAGE_MODEL_UNFREEZE = "layer.3|layer.4" 
    
    # Гиперпараметры
    BATCH_SIZE = 32
    TEXT_LR = 3e-5      #  LR для текстовой модели
    IMAGE_LR = 1e-4     #  LR для изображений
    CLASSIFIER_LR = 5e-4 #  LR для классификатора
    EPOCHS = 10
    DROPOUT = 0.15
    HIDDEN_DIM = 256 # размерность проекции признаков моделей
    NUM_CLASSES = 2

    # Пути
    TRAIN_DF_PATH = df_path
    VAL_DF_PATH = dataset_path / "val.csv"
    SAVE_PATH = BASE_DIR / "models" / "multimodal" / "best_model.pth" 

In [23]:
config = Config()

# Подготовка датасета

In [24]:
class MultimodalDataset(Dataset):

    def __init__(self, transforms, ds_type="train"):
        if ds_type == "train":
            self.df = pd.read_csv(config.TRAIN_DF_PATH)
        else:
            self.df = pd.read_csv(config.VAL_DF_PATH)
        self.image_cfg = timm.get_pretrained_cfg(config.IMAGE_MODEL_NAME)
        self.tokenizer = AutoTokenizer.from_pretrained(config.TEXT_MODEL_NAME)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx, "text"]
        label = self.df.loc[idx, "label"]

        img_path = self.df.loc[idx, "image_path"]
        image = Image.open(images_path / img_path).convert('RGB')
        image = self.transforms(image=np.array(image))["image"]

        return {"label": label, "image": image, "text": text}

In [25]:
def collate_fn(batch, tokenizer):
    texts = [item["text"] for item in batch]
    images = torch.stack([item["image"] for item in batch])
    labels = torch.LongTensor([item["label"] for item in batch])

    tokenized_input = tokenizer(texts,
                                return_tensors="pt",
                                padding="max_length",
                                truncation=True)
    return {
        "label": labels,
        "image": images,
        "input_ids": tokenized_input["input_ids"],
        "attention_mask": tokenized_input["attention_mask"]
    }

In [26]:
text_model = "bert-base-uncased"
image_model = 'tf_efficientnet_b0'
tokenizer = AutoTokenizer.from_pretrained(text_model)
cfg = timm.get_pretrained_cfg(image_model)

In [27]:
def get_transforms(ds_type="train"):
    cfg = timm.get_pretrained_cfg(config.IMAGE_MODEL_NAME)

    if ds_type == "train":
        transforms = A.Compose(
            [
                A.SmallestMaxSize(
                    max_size=max(cfg.input_size[1], cfg.input_size[2]), p=1.0),
                A.RandomCrop(
                    height=cfg.input_size[1], width=cfg.input_size[2], p=1.0),
                A.Affine(scale=(0.8, 1.2),
                        rotate=(-15, 15),
                        translate_percent=(-0.1, 0.1),
                        shear=(-10, 10),
                        fill=0,
                        p=0.8),
                A.CoarseDropout(num_holes_range=(2, 8),
                                hole_height_range=(int(0.07 * cfg.input_size[1]),
                                                int(0.15 * cfg.input_size[1])),
                                hole_width_range=(int(0.1 * cfg.input_size[2]),
                                                int(0.15 * cfg.input_size[2])),
                                fill=0,
                                p=0.5),
                A.ColorJitter(
                    brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.7),
                A.Normalize(mean=cfg.mean, std=cfg.std),
                A.ToTensorV2(p=1.0)
            ],
            seed=42,
        )
    else:
        transforms = A.Compose(
            [
                A.SmallestMaxSize(
                    max_size=max(cfg.input_size[1], cfg.input_size[2]), p=1.0),
                A.CenterCrop(
                    height=cfg.input_size[1], width=cfg.input_size[2], p=1.0),
                A.Normalize(mean=cfg.mean, std=cfg.std),
                A.ToTensorV2(p=1.0)
            ]
        )

    return transforms 

In [28]:
df = pd.read_csv(df_path)
ds = MultimodalDataset(transforms=get_transforms())

loader = DataLoader(ds,
                    batch_size=1,
                    shuffle=False,
                    collate_fn=partial(collate_fn, tokenizer=tokenizer)) 

# Модель

In [29]:
class MultimodalModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.text_model = AutoModel.from_pretrained(config.TEXT_MODEL_NAME).to(device)
        self.image_model = timm.create_model(
            config.IMAGE_MODEL_NAME,
            pretrained=True,
            num_classes=0 
        )
        self.image_model = self.image_model.to(device)

        self.text_proj = nn.Linear(self.text_model.config.hidden_size, config.HIDDEN_DIM)
        self.image_proj = nn.Linear(self.image_model.num_features, config.HIDDEN_DIM)

        self.classifier = nn.Sequential(
            nn.Linear(config.HIDDEN_DIM, config.HIDDEN_DIM // 2),   
            nn.LayerNorm(config.HIDDEN_DIM // 2),         
            nn.ReLU(),                           
            nn.Dropout(0.15),                    
            nn.Linear(config.HIDDEN_DIM // 2, config.NUM_CLASSES) 
        )

    def forward(self, input_ids, attention_mask, image):
        text_features = self.text_model(input_ids, attention_mask).last_hidden_state[:,  0, :]
        image_features = self.image_model(image)

        text_emb = self.text_proj(text_features)
        image_emb = self.image_proj(image_features)

        fused_emb = text_emb * image_emb
        
        logits = self.classifier(fused_emb)
        return logits

In [30]:
def set_requires_grad(module, unfreeze_pattern="", verbose=False):
    if len(unfreeze_pattern) == 0:
        for param, _ in module.named_parameters():
            param.requires_grad = False
        return

    pattern = re.compile(unfreeze_pattern)

    for name, param in module.named_parameters():
        if pattern.search(name):
            param.requires_grad = True
            if verbose:
                print(f"Разморожен слой: {name}")
        else:
            param.requires_grad = False

In [31]:
# Разморозка последнего слоя resnet50
image_model = timm.create_model(
            'resnet50',
            pretrained=True,
            num_classes=0
        )

set_requires_grad(image_model, unfreeze_pattern="layer4", verbose=True)

Разморожен слой: layer4.0.conv1.weight
Разморожен слой: layer4.0.bn1.weight
Разморожен слой: layer4.0.bn1.bias
Разморожен слой: layer4.0.conv2.weight
Разморожен слой: layer4.0.bn2.weight
Разморожен слой: layer4.0.bn2.bias
Разморожен слой: layer4.0.conv3.weight
Разморожен слой: layer4.0.bn3.weight
Разморожен слой: layer4.0.bn3.bias
Разморожен слой: layer4.0.downsample.0.weight
Разморожен слой: layer4.0.downsample.1.weight
Разморожен слой: layer4.0.downsample.1.bias
Разморожен слой: layer4.1.conv1.weight
Разморожен слой: layer4.1.bn1.weight
Разморожен слой: layer4.1.bn1.bias
Разморожен слой: layer4.1.conv2.weight
Разморожен слой: layer4.1.bn2.weight
Разморожен слой: layer4.1.bn2.bias
Разморожен слой: layer4.1.conv3.weight
Разморожен слой: layer4.1.bn3.weight
Разморожен слой: layer4.1.bn3.bias
Разморожен слой: layer4.2.conv1.weight
Разморожен слой: layer4.2.bn1.weight
Разморожен слой: layer4.2.bn1.bias
Разморожен слой: layer4.2.conv2.weight
Разморожен слой: layer4.2.bn2.weight
Разморожен 

# Обучение

In [32]:
def train():
    # Инициализация модели
    model = MultimodalModel().to(device)
    tokenizer = AutoTokenizer.from_pretrained(config.TEXT_MODEL_NAME)
    # Оптимизатор с разными LR
    optimizer = AdamW([
        {'params': model.text_model.parameters(), 'lr': config.TEXT_LR},
        {'params': model.image_model.parameters(), 'lr': config.IMAGE_LR},
        {'params': model.classifier.parameters(), 'lr': config.CLASSIFIER_LR}
    ])
    
    criterion = nn.CrossEntropyLoss()
    
    # Загрузка данных
    transforms = get_transforms(config)
    val_transforms = get_transforms(ds_type="val")
    train_dataset = MultimodalDataset(transforms)
    val_dataset = MultimodalDataset(val_transforms, ds_type="val")
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        collate_fn=partial(collate_fn, tokenizer=tokenizer)
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        collate_fn=partial(collate_fn, tokenizer=tokenizer)
    )
    
    # инициализируем метрику
    f1_metric = torchmetrics.F1Score(
        task="binary" if config.NUM_CLASSES == 2 else "multiclass", 
        num_classes=config.NUM_CLASSES).to(device)
    best_f1 = 0.0
    
    for epoch in range(config.EPOCHS):
        model.train()
        total_loss = 0.0
        
        for batch in train_loader:
            # Подготовка данных
            inputs = {
                'input_ids': batch['input_ids'].to(device),
                'attention_mask': batch['attention_mask'].to(device),
                'image': batch['image'].to(device)
            }
            labels = batch['label'].to(device)
            
            # Forward
            optimizer.zero_grad()
            logits = model(**inputs)
            loss = criterion(logits, labels)
            
            # Backward
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        # Валидация
        val_f1 = validate(model, val_loader, f1_metric)
        f1_metric.reset()
        
        print(f"Epoch {epoch+1}/{config.EPOCHS} | avg_Loss: {total_loss/len(train_loader):.4f} | Val F1: {val_f1 :.4f}")
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), config.SAVE_PATH)

In [33]:
def validate(model, val_loader, f1_metric):
    model.eval()
    
    with torch.no_grad():
        for batch in val_loader:
            inputs = {
                'input_ids': batch['input_ids'].to(device),
                'attention_mask': batch['attention_mask'].to(device),
                'image': batch['image'].to(device)
            }
            labels = batch['label'].to(device)
            
            logits = model(**inputs)
            _, predicted = logits.argmax(dim=1)
            _ = f1_metric(preds=predicted, target=labels)
    
    return f1_metric.compute().cpu().numpy() 

In [34]:
train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ValueError: too many dimensions 'str'